# 04 · Live viewer (A-line, envelope, M-mode)

`minus.live.LiveView` is a live object for Jupyter: run controls (start/stop, mode, input, gain) above an image that
updates in place while a background thread fires shots. Each frame shows the A-line (RF), its envelope in dB and an
M-mode image (the last 120 envelopes, depth vertical). Modes: `pe` (pulse + RF, ~22 frames/s with the SPI link and base64
output), `acq` (no pulse), `env` (envelope computed on the PIC, ~43 frames/s), `tof` (`pe` + the PIC's time of flight as a
red marker). The same figure runs as a desktop window: `python3 -m minus.live` (see `--help`).

**While a view is running it owns the serial port** — call `view.stop()` before using `dev` in other cells.

This notebook was executed on board 3 **without a piezo**: input 19 reads the PIC DAC triangle so the panels have
something to show. With a transducer, use input 25 (the default) and pick the gain with the dropdown.

In [1]:
import sys, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "library"))
from minus import Minus
from minus.live import LiveView
dev = Minus()
dev.dac("tri", 512, 3584, 16)       # test signal on input 19 (no piezo on this bench)

Dac(mode='tri', code=None, lo=512, hi=3584, slope=16, period_us=24.58, hz=None, points=None, update_us=None, amp=None, mid=None)

In [2]:
view = LiveView(dev, mode="pe", in_=19)   # displays the controls + image

/home/kelu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Synchronous frames (what a headless run, e.g. `nbconvert --execute`, can show):

In [3]:
f = view.run(frames=15)
print(view.frames, "frames;", f.info)

15 frames; pe in 19: 4095 samples, 40.2 ns, spi@12.50MHz 5713us


Background thread: the image keeps updating while the cell has returned (press **run** in the widget, or):

In [4]:
view.start(); time.sleep(3); view.stop()
print(view.frames, "frames so far;", view.w_status.value)

Exception in thread Thread-6 (_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/kelu/.local/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 761, in run_closure
    _threading_Thread_run(self)
  File "/usr/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/kelu/ultrasounds/minus/python/library/minus/live.py", line 231, in _loop
    self.w_run.value = False
  File "/home/kelu/.local/lib/python3.10/site-packages/traitlets/traitlets.py", line 716, in __set__
    self.set(obj, value)
  File "/home/kelu/.local/lib/python3.10/site-packages/traitlets/traitlets.py", line 706, in set
    obj._notify_trait(self.name, old_value, new_value)
  File "/home/kelu/.local/lib/python3.10/site-packages/traitlets/traitlets.py", line 1513, in _notify_trait
    self.notify_change(
  File "/home/kelu/.local/lib/python3.10/site-packages/ipywidgets/widg

30 frames so far; stopped after 30 frames


Envelope mode (the PIC sends 511 envelope values instead of 4095 samples):

In [5]:
view.src.mode = "env"; view.src.dec = 8
view.run(frames=10); print(view.w_status.value)

frame 40: env dec 8 x1 shots, 511 pts, spi@12.50MHz


In [6]:
view.stop()
dev.dac("off"); dev.close()